# Predicting Fetal Distress: Classifying Fetal Health from Cardiotocography Data

**Author:** Angela Mukami, Catherine Nanjala, Moffat Mwangi, Tafford Pessah

---

## 1. Business Understanding

Monitoring fetal health during pregnancy is critical to ensure both maternal and fetal well-being. One of the most common methods for evaluating fetal health is Cardiotocography (CTG), which measures fetal heart rate and uterine contractions. Abnormalities in CTG readings can indicate potential fetal distress, requiring immediate intervention to prevent complications during labour and delivery.

However, interpreting CTG data is complex and subject to inter-observer variability -- different clinicians can interpret the same trace differently. An automated classification system that reliably identifies fetal distress from CTG measurements could assist healthcare providers in making faster, more consistent decisions.

### The predictive question

> Using Cardiotocography (CTG) measurements -- fetal heart rate patterns, uterine contractions, decelerations, and variability indicators -- can we classify fetal health status as Normal, Suspected, or Abnormal?

This would be useful because early and accurate classification of fetal health enables timely clinical intervention for high-risk pregnancies, potentially reducing adverse outcomes for both mother and baby.

### Who would use this?

- **Healthcare providers**: An automated tool to assist in interpreting CTG data, enabling faster assessment of fetal health and more consistent decision-making across clinicians.
- **Medical researchers**: Exploring which CTG measurements are most strongly associated with fetal distress, advancing knowledge in obstetrics and maternal-fetal medicine.
- **Hospital systems**: Reducing unnecessary interventions by accurately distinguishing normal cases from those requiring action, while ensuring genuinely at-risk cases are not missed.

### Domain context

Cardiotocography has been a standard tool in obstetric care since the 1960s. While it is widely used, its interpretation remains subjective. Studies have shown significant inter-observer and intra-observer variability in CTG interpretation, which can lead to both over-intervention (unnecessary caesarean sections) and under-intervention (missed fetal distress). Machine learning approaches to CTG classification aim to reduce this variability by providing consistent, data-driven assessments.

A critical consideration in this domain is the asymmetry of errors. In clinical terms, a false negative on an Abnormal case -- classifying a distressed fetus as normal -- is far more dangerous than a false positive. This shapes our evaluation strategy: we prioritise recall on the Abnormal class over raw accuracy.

Source: Ayres-de-Campos, D., et al. (2015). FIGO consensus guidelines on intrapartum fetal monitoring: Cardiotocography. International Journal of Gynecology & Obstetrics, 131(1), 13-24.

## 2. Data Understanding

### 2.1 Imports and setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, recall_score
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 25)

print("All imports loaded successfully.")

All imports loaded successfully.


### 2.2 Load the dataset

The dataset contains 2,126 Cardiotocography (CTG) records. Each record includes 21 features measuring fetal heart rate patterns, uterine contractions, decelerations, variability indicators, and histogram-derived statistics. The target variable is fetal health status classified into three categories.

In [2]:
df = pd.read_csv('fetal_health.csv')
print(f"Dataset shape: {df.shape[0]} records, {df.shape[1]} columns")
df.head()

Dataset shape: 2126 records, 22 columns


,baseline value,accelerations,fetal_movement,uterine_contractions,light_decelerations,severe_decelerations,prolongued_decelerations,abnormal_short_term_variability,mean_value_of_short_term_variability,percentage_of_time_with_abnormal_long_term_variability,mean_value_of_long_term_variability,histogram_width,histogram_min,histogram_max,histogram_number_of_peaks,histogram_number_of_zeroes,histogram_mode,histogram_mean,histogram_median,histogram_variance,histogram_tendency,fetal_health
0,120.0,0.000,0.0,0.000,0.000,0.0,0.0,73.0,0.5,43.0,2.4,64.0,62.0,126.0,2.0,0.0,120.0,137.0,121.0,73.0,1.0,2.0
1,132.0,0.006,0.0,0.006,0.003,0.0,0.0,17.0,2.1,0.0,10.4,130.0,68.0,198.0,6.0,1.0,141.0,136.0,140.0,12.0,0.0,1.0
2,133.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.1,0.0,13.4,130.0,68.0,198.0,5.0,1.0,141.0,135.0,138.0,13.0,0.0,1.0
3,134.0,0.003,0.0,0.008,0.003,0.0,0.0,16.0,2.4,0.0,23.0,117.0,53.0,170.0,11.0,0.0,137.0,134.0,137.0,13.0,1.0,1.0
4,132.0,0.007,0.0,0.008,0.000,0.0,0.0,16.0,2.4,0.0,19.9,117.0,53.0,170.0,9.0,0.0,137.0,136.0,138.0,11.0,1.0,1.0
